# BBM 409 - Programming Assignment 2

**PART 1:** Binary Classification with SVM (30 points)  
**PART 2:** Multiclass Classification (70 points)


* You can add as many cells as you want in-between each question.
* Please add comments to your code to explain your work.  
* Please add Markdown cells to answer the (non-coding) questions in the homework text. You can, however, refer to the outputs of code cells without adding them as images to the Markdown cell unless you are requested to do otherwise.
* Please be careful about the order of runs of cells. Doing the homework, it is likely that you will be running the cells in different orders, however, they will be evaluated in the order they appear. Hence, please try running the cells in this order before submission to make sure they work.    
* Please refer to the homework text for any implementation detail. Though you are somewhat expected to abide by the comments in the below cells, they are mainly just provided for guidance. That is, as long as you are not completely off this structure and your work pattern is understandable and traceable, it is fine. For instance, you do not have to implement a particular function within a cell just because the comment directs you to do so.
* This document is also your report. Show your work.

###  Mustafa Kemal Öz 2230356179

## Import Required Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

---
# PART 1: BINARY CLASSIFICATION WITH SVM (30 POINTS)
---

## Load the Sonar Dataset from UCI Repository

In [3]:
print("Loading Sonar dataset...")
sonar = fetch_ucirepo(id=151)
X_sonar = sonar.data.features
y_sonar = sonar.data.targets
print("Features shape:", X_sonar.shape) # (208, 60)
print("Targets shape:", y_sonar.shape)  # (208, 1)

Loading Sonar dataset...
Features shape: (208, 60)
Targets shape: (208, 1)


## Encode the Target Labels using LabelEncoder()

In [4]:
# Encode target labels
le = LabelEncoder()
y_sonar_encoded = le.fit_transform(y_sonar.values.ravel())

# Display encoded classes and a sample of encoded targets
print("Encoded classes:", le.classes_)
print("Test", y_sonar_encoded[:5])

Encoded classes: ['M' 'R']
Test [1 1 1 1 1]


## Split the Data into 80% Training and 20% Testing Sets

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sonar,               # Features
    y_sonar_encoded,       # Labels
    test_size = 0.2,         # Test set ratio 20%
    stratify = y_sonar_encoded, # Preserve class ratios (Stratification)
    random_state = 22        # For reproducibility
)

# Check the sizes of the splits
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (166, 60)
Test set size: (42, 60)


---
## 1.1. Linear Kernel SVM without Hyperparameter Tuning (5 points)
---

## Create a Pipeline with StandardScaler and Linear SVM

In [6]:
pipeline_linear = Pipeline([
    ('scaler', StandardScaler()),      # Standardize the features
    ('svm', SVC(kernel='linear'))      # Build Linear Kernel SVM Model
])

## Train the Linear SVM Model

In [7]:
pipeline_linear.fit(X_train, y_train)

,steps,"[('scaler', ...), ('svm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'


## Predict on the Test Set and Calculate Accuracies and Print it

In [8]:
y_pred_train = pipeline_linear.predict(X_train) # Training accuracy
y_pred_test = pipeline_linear.predict(X_test)   # Test accuracy

In [9]:
print("Linear SVM Results (No Tuning):")
print("-" * 30)
print(f"Training Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:     {accuracy_score(y_test, y_pred_test):.4f}")

Linear SVM Results (No Tuning):
------------------------------
Training Accuracy: 0.9518
Test Accuracy:     0.7619


## Display Classification Report for Linear SVM

In [10]:
print("\nClassification Report (Test):\n")
print(classification_report(y_test, y_pred_test))


Classification Report (Test):

              precision    recall  f1-score   support

           0       0.71      0.91      0.80        22
           1       0.86      0.60      0.71        20

    accuracy                           0.76        42
   macro avg       0.79      0.75      0.75        42
weighted avg       0.78      0.76      0.76        42



## Display Confusion Matrix for Linear SVM

In [11]:
print("\nConfusion Matrix (Test):\n")
print(confusion_matrix(y_test, y_pred_test))


Confusion Matrix (Test):

[[20  2]
 [ 8 12]]


---
## 1.2. SVM with GridSearchCV and 5-Fold Cross-Validation (15 points)
---

## Create a Pipeline for GridSearchCV

In [12]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

## Define the Parameter Grid for Different Kernels

In [13]:
param_grid = [
    # Linear Kernel
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1, 10, 100]
    },
    # RBF Kernel
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 'auto', 0.001, 0.01]
    },
    # Polynomial Kernel
    {
        'svm__kernel': ['poly'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 'auto'],
        'svm__degree': [2, 3, 4]
    }
]

## Create Stratified 5-Fold Cross-Validation

In [14]:
cross_validation = StratifiedKFold(n_splits=5)

## Initialize and Run GridSearchCV on Training Data

In [15]:
# Initialize
grid_search = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_grid,
    cv=cross_validation,            # 5-Fold CV
    scoring='accuracy',             # Evaluation metric
    n_jobs=-1,                      # Use all processors
    verbose=1                       # Show progress
)

# Fit Grid Search
print("Grid Search Started... (This may take a while)")
grid_search.fit(X_train, y_train)

Grid Search Started... (This may take a while)
Fitting 5 folds for each of 44 candidates, totalling 220 fits


,estimator,"Pipeline(step...svm', SVC())])"
,param_grid,"[{'svm__C': [0.1, 1, ...], 'svm__kernel': ['linear']}, {'svm__C': [0.1, 1, ...], 'svm__gamma': ['scale', 'auto', ...], 'svm__kernel': ['rbf']}, ...]"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo...shuffle=False)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


## Display Best Parameters and Cross-Validation Score

In [16]:
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_}")


Best Parameters: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Best Cross-Validation Score: 0.8614973262032086


## Evaluate the Best Model on the Test Set

In [17]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\n--- Test Set Results (Tuned Model) ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")


--- Test Set Results (Tuned Model) ---
Test Accuracy: 0.9048


## Display Top 5 Parameter Combinations

In [18]:
results_df = pd.DataFrame(grid_search.cv_results_)
# Sort by score and take the top 5
top_5 = results_df.sort_values(by='mean_test_score', ascending=False).head(5)

print("\n--- Top 5 Parameter Combinations ---")
# Select and display relevant columns
columns_to_show = ['param_svm__kernel', 'param_svm__C', 'param_svm__gamma', 'param_svm__degree', 'mean_test_score', 'std_test_score']
print(top_5[columns_to_show].to_string(index=False))


--- Top 5 Parameter Combinations ---
param_svm__kernel  param_svm__C param_svm__gamma  param_svm__degree  mean_test_score  std_test_score
              rbf          10.0             auto                NaN         0.861497        0.059226
              rbf         100.0             auto                NaN         0.861497        0.059226
              rbf         100.0            scale                NaN         0.861497        0.059226
              rbf          10.0            scale                NaN         0.861497        0.059226
              rbf          10.0             0.01                NaN         0.855258        0.070417


## Display Classification Report for Tuned SVM

In [19]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best))


Classification Report:

              precision    recall  f1-score   support

           0       0.85      1.00      0.92        22
           1       1.00      0.80      0.89        20

    accuracy                           0.90        42
   macro avg       0.92      0.90      0.90        42
weighted avg       0.92      0.90      0.90        42



## Display Confusion Matrix for Tuned SVM

In [20]:
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_best))


Confusion Matrix:

[[22  0]
 [ 4 16]]


## Compare Linear SVM vs Tuned SVM Results. Compare experimental results. Explain the performance impact of linear and nonlinear kernels. Why is kernel trick important? Also explain why using k-fold cross validation is more advantageous than train-test split. (10 points)

### Comparison and Analysis

The experimental results demonstrate that the **Tuned SVM** significantly outperformed the baseline **Linear SVM** (Test Accuracy: **0.9048** vs. **0.7619**). This performance gap indicates that the Sonar dataset contains complex, non-linear patterns that a simple linear hyperplane cannot capture. By utilizing **nonlinear kernels** (like RBF) through hyperparameter tuning, the model created flexible decision boundaries that better adapted to the data's underlying structure, whereas the Linear SVM suffered from underfitting.

The **Kernel Trick** was crucial for this success, as it allowed the model to efficiently find these separators in high-dimensional spaces without the computational cost of explicit transformation. Furthermore, employing **k-Fold Cross-Validation** was far more advantageous than a simple train-test split for this small dataset (208 samples). By averaging performance across multiple folds, k-Fold CV eliminated the variance caused by random splitting (the "luck factor") and ensured that the selected hyperparameters were robust and generalizable, rather than being optimized for a single, potentially biased subset of data.

---
# PART 2: MULTICLASS CLASSIFICATION (70 POINTS)
---

## Load the Dry Beans Dataset from UCI Repository

In [21]:
print("Loading Dry Beans dataset...")
dry_bean = fetch_ucirepo(id=602)
X_beans = dry_bean.data.features
y_beans = dry_bean.data.targets
    
print("Features shape:", X_beans.shape)
print("Targets shape:", y_beans.shape)

Loading Dry Beans dataset...
Features shape: (13611, 16)
Targets shape: (13611, 1)


## Encode the Target Labels using LabelEncoder

In [22]:
# Encode target labels
le_beans = LabelEncoder()
y_beans_encoded = le_beans.fit_transform(y_beans.values.ravel())

print("Encoded classes:", le_beans.classes_)
print("First 5 labels encoded:", y_beans_encoded[:5])

Encoded classes: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']
First 5 labels encoded: [5 5 5 5 5]


## Split the Data into 80% Training and 20% Testing Sets

In [23]:
# 1. Stratified Train-Test Split (%80 Train, %20 Test)
X_train_beans, X_test_beans, y_train_beans, y_test_beans = train_test_split(
    X_beans,               
    y_beans_encoded,
    test_size=0.2, 
    stratify=y_beans_encoded,
    random_state=15
)

print(f"Train Shape: {X_train_beans.shape}")
print(f"Test Shape: {X_test_beans.shape}")

Train Shape: (10888, 16)
Test Shape: (2723, 16)


## Scale the Features using StandardScaler

In [25]:
scaler_beans = StandardScaler()

X_train_scaled = scaler_beans.fit_transform(X_train_beans)
X_test_scaled = scaler_beans.transform(X_test_beans)

---
## 2.1. Multinomial Logistic Regression (20 points)
---

## Define the Multinomial Logistic Regression Class

In [26]:
from sklearn.base import BaseEstimator, ClassifierMixin

class MultinomialLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, learning_rate=0.01, epochs=500, reg_lambda=0.01):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.weights = None
        self.bias = None
        self.classes_ = None
        self.loss_history = []

    def _softmax(self, z):
        z_safe = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z_safe)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def _cross_entropy(self, y_true_encoded, y_prob):
        # error prevention for log(0)
        epsilon = 1e-15
        y_prob = np.clip(y_prob, epsilon, 1 - epsilon)
        
        # Basic Loss
        loss = -np.mean(np.sum(y_true_encoded * np.log(y_prob), axis=1))
        
        # L2 Regularization: (lambda / 2) * ||w||^2
        l2_loss = (self.reg_lambda / 2) * np.sum(self.weights ** 2)
        
        return loss + l2_loss

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # 1. Initialize Weights and Bias
        self.weights = np.zeros((n_features, n_classes))
        self.bias = np.zeros(n_classes)

        # 2. Convert target variable (y) to One-Hot Encoding format
        # Example: 1 -> [0, 1, 0, 0...]
        y_encoded = np.zeros((n_samples, n_classes))
        for i, c in enumerate(self.classes_):
            y_encoded[y == c, i] = 1

        # 3. Gradient Descent Loop
        for epoch in range(self.epochs):
            # A. Linear Model: z = X * w + b
            z = np.dot(X, self.weights) + self.bias
            
            # B. Find prediction probabilities (y_prob) using Softmax
            y_prob = self._softmax(z)
            
            # C. Calculate Gradient
            # Error difference: (Prediction - Actual)
            error = y_prob - y_encoded
            
            # Weight derivative: (1/N) * X.T * error + (lambda * w)
            dw = (1 / n_samples) * np.dot(X.T, error) + (self.reg_lambda * self.weights)
            
            # Bias derivative
            db = (1 / n_samples) * np.sum(error, axis=0)

            # D. Update Parameters (Move in the opposite direction of the gradient by the learning rate)
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Save Loss value every 100 epochs
            if epoch % 100 == 0:
                loss = self._cross_entropy(y_encoded, y_prob)
                self.loss_history.append(loss)

        return self

    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self._softmax(z)

    def predict(self, X):
        y_prob = self.predict_proba(X)
        indices = np.argmax(y_prob, axis=1)
        return self.classes_[indices]

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

## Define Hyperparameter Grid for Multinomial Logistic Regression

In [27]:
param_grid_mlr = {
    'learning_rate': [0.01, 0.05, 0.1, 0.5],
    'epochs': [200, 500, 1000],
    'reg_lambda': [0.0, 0.01, 0.1]
}

## Run GridSearchCV for Multinomial Logistic Regression

In [28]:
mlr = MultinomialLogisticRegression()

grid_mlr = GridSearchCV(
    estimator=mlr,
    param_grid=param_grid_mlr,
    cv=StratifiedKFold(n_splits=5), # 5-Fold Cross Validation
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Multinomial Logistic Regression Grid Search Started...")
grid_mlr.fit(X_train_scaled, y_train_beans)
    
print(f"\nBest Parameters: {grid_mlr.best_params_}")
print(f"Best CV Score: {grid_mlr.best_score_:.4f}")

Multinomial Logistic Regression Grid Search Started...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Parameters: {'epochs': 1000, 'learning_rate': 0.5, 'reg_lambda': 0.0}
Best CV Score: 0.9223


## Evaluate Multinomial Logistic Regression on Test Set

In [29]:
best_mlr = grid_mlr.best_estimator_
y_pred_mlr = best_mlr.predict(X_test_scaled)

print("--- Multinomial Logistic Regression Test Results ---")
print(f"Test Accuracy: {accuracy_score(y_test_beans, y_pred_mlr):.4f}")

--- Multinomial Logistic Regression Test Results ---
Test Accuracy: 0.9251


## Display Classification Report for Multinomial Logistic Regression

In [30]:
print("\nClassification Report:\n")
print(classification_report(y_test_beans, y_pred_mlr, target_names=le_beans.classes_))


Classification Report:

              precision    recall  f1-score   support

    BARBUNYA       0.94      0.89      0.92       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.96      0.95       326
    DERMASON       0.93      0.92      0.92       709
       HOROZ       0.94      0.96      0.95       386
       SEKER       0.94      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.93      2723
   macro avg       0.94      0.94      0.94      2723
weighted avg       0.93      0.93      0.93      2723



## Display Confusion Matrix for Multinomial Logistic Regression

In [31]:
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test_beans, y_pred_mlr))


Confusion Matrix:

[[236   0  16   0   1   4   8]
 [  0 104   0   0   0   0   0]
 [  7   0 314   0   3   0   2]
 [  0   0   0 651   3  11  44]
 [  1   0   4   5 370   0   6]
 [  5   0   0   7   0 381  13]
 [  1   0   1  39  15   8 463]]


---
## 2.2. Decision Tree (20 points)
---

###  Define the Decision Tree model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix. (15 points)

## Define Hyperparameter Grid for Decision Tree

In [32]:
dt_model = DecisionTreeClassifier(random_state=15)

param_grid_dt = {
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

grid_dt = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid_dt,
    cv=StratifiedKFold(n_splits=5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

## Run GridSearchCV for Decision Tree

In [33]:
print("Decision Tree Grid Search Started...")
grid_dt.fit(X_train_scaled, y_train_beans)

print(f"\nBest Parameters: {grid_dt.best_params_}")
print(f"Best CV Score: {grid_dt.best_score_:.4f}")

Decision Tree Grid Search Started...
Fitting 5 folds for each of 90 candidates, totalling 450 fits

Best Parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10}
Best CV Score: 0.9051


## Display Classification Report for Decision Tree

In [34]:
best_dt = grid_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test_scaled)

# 2. Sonuçları Raporla [cite: 129]
print("--- Decision Tree Test Results ---")
print(f"Test Accuracy: {accuracy_score(y_test_beans, y_pred_dt):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_beans, y_pred_dt, target_names=le_beans.classes_))

--- Decision Tree Test Results ---
Test Accuracy: 0.9130

Classification Report:

              precision    recall  f1-score   support

    BARBUNYA       0.90      0.87      0.88       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.89      0.95      0.92       326
    DERMASON       0.91      0.93      0.92       709
       HOROZ       0.97      0.91      0.94       386
       SEKER       0.96      0.92      0.94       406
        SIRA       0.85      0.87      0.86       527

    accuracy                           0.91      2723
   macro avg       0.93      0.92      0.92      2723
weighted avg       0.91      0.91      0.91      2723



## Display Confusion Matrix for Decision Tree

In [35]:
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test_beans, y_pred_dt))


Confusion Matrix:

[[230   0  22   0   2   4   7]
 [  0 104   0   0   0   0   0]
 [ 13   0 310   0   2   0   1]
 [  0   0   0 658   1   8  42]
 [  1   0  11   5 353   0  16]
 [  4   0   0  15   0 375  12]
 [  7   0   6  46   7   5 456]]


###  Describe how a decision tree builds its decision structure. (5 points)

A Decision Tree constructs its decision structure through a recursive process called partitioning, starting from a root node that encompasses the entire dataset. At each step, the algorithm evaluates all available features to identify the optimal split that best separates the data into distinct classes, utilizing mathematical metrics like **Gini Impurity** or **Entropy** to measure and maximize the purity of the resulting groups. This process creates internal nodes representing feature tests and branches representing outcomes, continuing iteratively until specific stopping criteria—such as a maximum tree depth (`max_depth`) or minimum samples per leaf—are met, ultimately terminating in leaf nodes that assign the final class labels.

---
## 2.3. XGBoost (20 points)
---

###  Define the XGBoost model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix(10 points)

## Define Hyperparameter Grid for XGBoost

In [36]:
xgb_model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)

param_grid_xgb = {
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

## Run GridSearchCV for XGBoost

In [37]:
grid_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=StratifiedKFold(n_splits=5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

## Evaluate XGBoost on Test Set

In [38]:
print("XGBoost Grid Search Started... (This process may take some time)")
grid_xgb.fit(X_train_scaled, y_train_beans)

print(f"\nBest Parameters: {grid_xgb.best_params_}")
print(f"Best CV Score: {grid_xgb.best_score_:.4f}")

XGBoost Grid Search Started... (This process may take some time)
Fitting 5 folds for each of 144 candidates, totalling 720 fits

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100, 'subsample': 0.8}
Best CV Score: 0.9259


## Display Classification Report for XGBoost

In [39]:
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test_scaled)

print(f"Test Accuracy: {accuracy_score(y_test_beans, y_pred_xgb):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_beans, y_pred_xgb, target_names=le_beans.classes_))

Test Accuracy: 0.9354

Classification Report:

              precision    recall  f1-score   support

    BARBUNYA       0.94      0.92      0.93       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.95      0.96      0.96       326
    DERMASON       0.93      0.93      0.93       709
       HOROZ       0.95      0.96      0.95       386
       SEKER       0.96      0.95      0.95       406
        SIRA       0.90      0.89      0.89       527

    accuracy                           0.94      2723
   macro avg       0.95      0.94      0.95      2723
weighted avg       0.94      0.94      0.94      2723



## Display Confusion Matrix for XGBoost

In [40]:
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test_beans, y_pred_xgb))


Confusion Matrix:

[[244   0  12   1   1   4   3]
 [  0 104   0   0   0   0   0]
 [  9   0 313   0   3   0   1]
 [  0   0   0 659   4  10  36]
 [  1   0   4   4 371   0   6]
 [  3   0   0   8   0 386   9]
 [  2   0   0  38  13   4 470]]


### What are the revolutionary features of XGBoost compared to other tree-based models? (10 points)

**XGBoost (Extreme Gradient Boosting)** revolutionizes tree-based modeling primarily through its regularized objective function, which adds a penalty term **Ω(ft)** to the standard loss function **(Obj=∑L+∑Ω)**. Unlike traditional Gradient Boosting Machines (GBM), this regularization effectively controls model complexity and significantly reduces the risk of overfitting. Furthermore, XGBoost distinguishes itself with system optimization features such as parallel processing during node splitting, which drastically reduces training time compared to sequential GBMs, and algorithmic enhancements like using second-order gradients **(Newton-Raphson method)** for more precise loss minimization, built-in handling of missing values, and an advanced depth-first tree pruning strategy.

###  Compare the classification results of your Multinomial Logistic Regression, Decision Tree, and XGBoost models on the Dry Beans dataset. Discuss the comparison in terms of overall model performance, risk of overfitting, model complexity and the scenarios in which each model is most effective. (10 points)

### Comparison of Classification Models on Dry Beans Dataset

#### 1. Overall Model Performance
* **XGBoost:** As expected, XGBoost achieved the highest performance (Test Accuracy: **0.9354**). Its ensemble nature, which sequentially corrects errors made by previous trees, allowed it to capture complex interactions between bean features better than any single model.
* **Multinomial Logistic Regression:** Surprisingly, the linear MLR model outperformed the Decision Tree (Test Accuracy: **0.9251**). This suggests that the decision boundaries between dry bean classes are largely linearly separable, or that the L2 regularization in our implementation provided better generalization than a single Decision Tree.
* **Decision Tree:** The Decision Tree achieved the lowest performance among the three (Test Accuracy: **0.9130**). While theoretically capable of capturing non-linear patterns, a single tree often suffers from high variance. Even with hyperparameter tuning, it could not match the stability of the regularized linear model or the boosting power of XGBoost.

#### 2. Risk of Overfitting
* **Decision Tree:** This model has the **highest risk of overfitting**. Without strict constraints (like `max_depth`), a decision tree tends to memorize training noise. In our experiment, despite pruning via GridSearchCV, its lower test accuracy compared to MLR suggests it might have slightly overfitted the training data or failed to generalize as well as the linear model.
* **XGBoost:** XGBoost has a **low-to-moderate risk**. Unlike standard decision trees, XGBoost explicitly includes regularization terms (L1/L2) in its objective function ($\Omega(f_t)$) to penalize complexity. The `subsample` and `colsample_bytree` parameters further reduce overfitting by introducing randomness.
* **MLR:** This model has the **lowest risk of overfitting** due to its high bias (simplicity). The L2 regularization (`reg_lambda`) we implemented further prevented the weights from becoming too large, likely contributing to its superior performance over the Decision Tree in this specific experiment.

#### 3. Model Complexity
* **MLR (Lowest):** Computationally efficient and fast. Consists only of weights and biases. Highly interpretable through coefficients.
* **Decision Tree (Medium):** Moderate complexity. Faster training than XGBoost but slower than MLR. Visually interpretable, but deep trees can become complex and harder to trace.
* **XGBoost (Highest):** Most complex and computationally expensive. Requires training hundreds of trees sequentially. Considered a "black box"; while feature importance is available, tracing decisions across the ensemble is difficult.

#### 4. Scenarios for Effectiveness
* **Multinomial Logistic Regression:** Best used when **speed, simplicity, and interpretability** are priorities, or when data is linearly separable (as partially observed here). Ideal for real-time systems with limited resources.
* **Decision Tree:** Effective when **visual explainability** is required for non-technical stakeholders. Useful when feature scaling is not possible/desirable.
* **XGBoost:** The go-to choice for **maximizing accuracy** on structured tabular data. Most effective in competitions or high-stakes applications where a small increase in accuracy (like the ~1% gain over MLR here) justifies the extra computational cost.